In [0]:
%pip install --pre databricks-feature-engineering>=0.13.1a4
dbutils.library.restartPython()

In [0]:
from datetime import timedelta

from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import (
    DeltaTableSource,
    ContinuousWindow,
    TumblingWindow,
    SlidingWindow,
    Sum,
    Avg,
    Count,
    Min,
    Max,
    StddevPop,
    ApproxCountDistinct,
)
from pyspark.sql import functions as F
import mlflow
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import DeltaTableSource, Sum, Avg, TumblingWindow, SlidingWindow , Count
from datetime import timedelta

In [0]:
fe = FeatureEngineeringClient()

CATALOG_NAME = "ananyaroy"
SCHEMA_NAME = "boq_mlops"
TABLE_NAME = "delinquency_history"

# 1. Create data source
source = DeltaTableSource(
    catalog_name=CATALOG_NAME,
    schema_name=SCHEMA_NAME,
    table_name=TABLE_NAME,
    entity_columns=["account_id"],
    timeseries_column="observation_date"
)

# Feature 2: Total clicks in last 30 days (SlidingWindow + Count)
acct_delinq_180d = fe.create_feature(
    catalog_name=CATALOG_NAME,
    schema_name=SCHEMA_NAME,
    name="count_of_delinquency",
    description="Total number of times customer became delinquent ",
    source=source,
    inputs=["delinquency_bucket"],
    function=Count(),
    time_window=SlidingWindow(
        window_duration=timedelta(days=180),
        slide_duration=timedelta(days=1)
    )
)

In [0]:
from databricks.feature_engineering.entities import OfflineStoreConfig

# Get feature objects for user engagement features
acct_delinq_feature_objects = []
for feature_name in ["count_of_delinquency"]:
    full_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{feature_name}"
    try:
        feature = fe.get_feature(full_name=full_name)
        acct_delinq_feature_objects.append(feature)
    except Exception as e:
        print(f"Failed to get {feature_name}: {e}")

# Configure offline store for user engagement features
acct_delinq_offline_store = OfflineStoreConfig(
    catalog_name=CATALOG_NAME,
    schema_name=SCHEMA_NAME,
    table_name_prefix="acct_delinq"
)

# Materialize user engagement features
fe.materialize_features(
    features=acct_delinq_feature_objects,
    offline_config=acct_delinq_offline_store,
    pipeline_state="ACTIVE",
    cron_schedule="0 0 * * * ?"  # Daily at midnight
)

print(f"✓ Materialized {len(acct_delinq_feature_objects)} user engagement features with prefix 'user_engagement'")